# Reset all project tables
Destructive administrative notebook for a complete project restart. It truncates every existing Bronze, Silver, Gold, datamart, and operations table. It does not delete the uploaded Parquet files, External Volumes, schemas, or table definitions.

In [0]:
from pathlib import Path
import sys

#Path.cwd() : the current dir that contains the notebook
source_root = next(
    (
        root / "src" 
        for root in (Path.cwd(), *Path.cwd().parents) 
        if (root / "src").is_dir()
    )
    , None
)

if source_root is None:
    #if don't find the 'src' folder, raise an error
    raise FileNotFoundError("Open this notebook from the FinOps Cloud Data Platform Git Folder.")

if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

In [0]:
ENVIRONMENT = "dev"
CONFIRM_RESET = ""
dbutils.widgets.dropdown("environment", ENVIRONMENT, ["dev", "prod"])
dbutils.widgets.text("confirm_reset", CONFIRM_RESET)
ENVIRONMENT = dbutils.widgets.get("environment")
CONFIRM_RESET = dbutils.widgets.get("confirm_reset")

if CONFIRM_RESET != "RESET":
    raise ValueError("Reset blocked: set confirm_reset exactly to RESET.")

In [0]:
"""

src/
└── finops_cloud/
    ├── config.py
    │      └── load_config()
    │
    ├── maintenance.py
    │      └── truncate_project_tables()
    │
    └── runtime.py
           └── get_spark()


"""


In [0]:
from finops_cloud.config import load_config
from finops_cloud.maintenance import truncate_project_tables
from finops_cloud.runtime import get_spark

config = load_config(ENVIRONMENT)
spark_session = get_spark(config.profile)
truncated = truncate_project_tables(spark_session, config)


dbutils.jobs.taskValues.set(key="truncated_count", value=len(truncated))

print(f"RESET COMPLETED: {len(truncated)} table(s) truncated in {ENVIRONMENT}.")

for table in truncated:
    print(f"- {table}")
    